In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
from PIL import Image
from io import BytesIO
import re
import json
from qwen_vl_utils import process_vision_info
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from fastprogress import progress_bar
from sklearn.metrics import f1_score, precision_score, recall_score
import torch
from io import BytesIO
import h5py
import json
from PIL import Image

KeyboardInterrupt: 

In [ ]:
def prompt_selection(baseline=True, guide_dict=None):
    if baseline:
        prompt = """\
        Detect ranges of anomalies in this time series, in terms of the x-axis coordinate.
        List one by one, in JSON format. 
        If there are no anomalies, answer with an empty list [].

        Output template:
        [{"start": ..., "end": ...}, {"start": ..., "end": ...}...]
        please do not provide any additional text or explanation.
        """

    # two guides inference
    else:
        prompt = f"""
        You are given three images.

        The first image serves as a visual guide, depicting a normal time series pattern based on the following description:
        {{
            "trend": {guide_dict['trend']},
            "seasonality": {guide_dict['seasonality']},
            "shape": {guide_dict['shape']},
            "state": {guide_dict['state']},
        }} 

        The second image is a visual guide showing four types of univariate time series anomalies.
        Each row corresponds to one anomaly type (from top to bottom): 
        (1) Frequency anomaly: unexpected changes in periodic patterns. 
        (2) Trend anomaly: changes in the gradient of the time series, such as acceleration, deceleration, or reversal. 
        (3) Point anomaly: individual points deviating significantly from the surrounding pattern. 
        (4) Out-of-range anomaly: values that strongly deviate from the normal range. 
        Green regions in the second image indicate the anomaly ranges for each type.

        The third image shows a univariate time series for anomaly detection.    

        Identify time ranges in the third image that deviate from the normal pattern shown in the first image, or that match any of the four anomaly types illustrated in the second image.  
        Return the detected anomaly ranges as a list of dictionaries, one per anomaly, in terms of the x-axis coordinate.
        If there are no anomalies, return an empty list [].
        
        Output format only: [{{"start": ..., "end": ...}}, {{"start": ..., "end": ...}}...] 
        Do not provide any additional explanation or text.
        """
    return prompt


def qwen_make_messages_one_image(prompt, image):
    messages = [
        {
            "role": "system", 
            "content": "You are a time series anomaly detector."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text", 
                    "text": prompt
                },
            ],
        }
    ]
    return messages


def qwen_make_messages_images(prompt, image1, image2, image3):
    messages = [
        {
            "role": "system", 
            "content": "You are a time series anomaly detector."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image1,
                },
                {
                    "type": "image",
                    "image": image2,
                },
                {
                    "type": "image",
                    "image": image3,
                },
                {
                    "type": "text", 
                    "text": prompt
                },
            ],
        }
    ]
    return messages


def qwen_inference(model, processor, messages, device='cuda'):
    # Preparation for inference
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(device)

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

In [ ]:
def get_data(data_path):
    with open(data_path, 'rb') as f:
        data = pickle.load(f)
        value_list = []
        label_list = []

        data_length = len(data['series'])

        for num in range(data_length):
            org_values = data['series'][num]
            values = [value[0] for value in org_values]

            org_answers = data['anom'][num]
            labels = [0 for _ in range(len(values))]
            for answer in org_answers[0]:
                start, end = answer[0], answer[1] # interval 
                labels[start:end] = [1 for _ in range(start, end)]

            value_list.append(values)
            label_list.append(labels)

    return value_list, label_list

def show_pil_image(image):
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    plt.close()
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

def make_ts_image(length, values, label=None, show_gt=False):
    plt.clf()
    plt.figure(figsize=(12, 2))
    plt.plot([num for num in range(length)],[score for score in values]) 
    if show_gt:
        anomalies_idx = [i for i,l in enumerate(label) if l==1] 
        plt.bar(anomalies_idx, 2, bottom=-1, width=1, color='green',alpha=0.5, label='Ground-truth') 
    plt.tight_layout()
    
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight') 
    buf.seek(0)
    plt.close()
    pil_image = Image.open(buf)
    return pil_image

def make_ts_image_for_testing(length, values, predicted, label, precision, recall, f1score):
    predicted_idx = [i for i,l in enumerate(predicted) if l==1] 
    anomalies_idx = [i for i,l in enumerate(label) if l==1] 

    plt.clf()
    plt.figure(figsize=(24, 4))
    plt.plot([num for num in range(length)],[score for score in values]) 
    plt.bar(predicted_idx, 1, bottom=0, width=1, color='red',alpha=0.5, label=f'Predicted') 
    plt.bar(anomalies_idx, 2, bottom=-1, width=1, color='green',alpha=0.5, label='Ground-truth') 
    plt.plot([], [], ' ', label=f'Precision: {precision:.2f}')
    plt.plot([], [], ' ', label=f'Recall: {recall:.2f}')
    plt.plot([], [], ' ', label=f'F1 Score: {f1score:.2f}')
    plt.xlabel("Time", fontsize=20)
    plt.ylabel("Value", fontsize=20)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)

    plt.legend(fontsize=24, loc='upper left')
    plt.tight_layout()
    
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight') 
    buf.seek(0)
    plt.close()
    pil_image = Image.open(buf)
    return pil_image

def evaluation(gt_labels, pred_labels):
    f1 = f1_score(gt_labels, pred_labels)
    precision = precision_score(gt_labels, pred_labels)
    recall = recall_score(gt_labels, pred_labels)
    return f1, precision, recall

In [ ]:
data_type = 'range'
data_path = f'/home/inpyo/inpyo/Code-LMTAD-main/data/synthetic/{data_type}/eval/data.pkl'
value_list, label_list = get_data(data_path)     
test_num = 70
test_length = len(label_list[test_num])
test_label = label_list[test_num]
test_values = value_list[test_num]
test_image = make_ts_image(test_length, test_values, test_label, False)
show_pil_image(test_image)

In [ ]:
caption_file_path = "Guide/range/normal_range/1000/1000_0.json"
with open(caption_file_path, 'r', encoding='utf-8') as f:
    normal_guide_caption = json.load(f)
two_guides_prompt = prompt_selection(baseline=False, guide_dict = normal_guide_caption)
print(two_guides_prompt)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

qwen_path = "/home/inpyo/inpyo/Code-LMTAD-main/llm_model/Qwen2-VL-7B-Instruct" #qwen 절대경로 입력필요
model = Qwen2VLForConditionalGeneration.from_pretrained(qwen_path,torch_dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained(qwen_path)

In [ ]:
data_type = 'range'
normal_guide_path = f'/home/inpyo/inpyo/Code-LMTAD-main/Guide/{data_type}/normal_feature.h5'
normal_caption_path = f'/home/inpyo/inpyo/Code-LMTAD-main/Guide/{data_type}/normal_guide.json'

base_path = "Guide/range/normal_range/1000"
file_name = "1000_0"

caption_file_path = f"{base_path}/{file_name}.json"
image_file_path = f"{base_path}/{file_name}.png"

with open(caption_file_path, 'r', encoding='utf-8') as f:
    normal_guide_caption = json.load(f)
print(f"캡션 로드 완료: {caption_file_path}")

normal_guide_image = Image.open(image_file_path)
print(f"이미지 로드 완료: {image_file_path}")


def get_path_range_set(json_file_path='Guide/anomaly_guide.json'):
    path_range_set = []
    for i in range(20):
        range_list = []
        length = 50*(i+1)
        path = f'Guide/anomaly_guide_{50*(i+1)}.png'
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            item = data[(length//50)-1]
            for j in range(4):
                guide_range = json.loads(item[f'range_{j+1}'])
                range_list.append(guide_range)
        path_range_set.append((path, range_list))
    return path_range_set

try:
    all_anomaly_guides = get_path_range_set() 
    index = (test_length // 50) - 1 
    anomaly_guide_image_path = all_anomaly_guides[index][0] 
    
    anomaly_guide_image = Image.open(anomaly_guide_image_path)
    print(f"이상 현상 가이드 이미지 로드 완료: {anomaly_guide_image_path}")


except Exception as e:
    print(f"!!! Anomaly Guide 로드 중 치명적 에러 발생: {e}")
    raise


In [ ]:

pil_image = make_ts_image(test_length, test_values, label=None, show_gt=False)
two_guides_prompt = prompt_selection(baseline=False, guide_dict=normal_guide_caption)

messages = qwen_make_messages_images(two_guides_prompt, normal_guide_image, anomaly_guide_image, pil_image)
output = qwen_inference(model, processor, messages)
try:
    output = re.findall(r'\[.*?\]', output)[0]
except:
    output = '[]'

# parsing
output_list = []
parsed = json.loads(output)
for item in parsed:
    output_list.append((item["start"], item["end"]))

predicted = [0 for _ in range(test_length)]
for answer in output_list:
    start, end = answer[0], answer[1] # interval
    predicted[start:end] = [1 for _ in range(start, end)]

# evaluation
f1, precision, recall = evaluation(test_label, predicted)
print('precision:', precision)
print('recall:', recall)
print('f1_score:', f1)

# visualization
pred_image = make_ts_image_for_testing(length=test_length, values=test_values, predicted=predicted, label=test_label, precision=precision, recall=recall, f1score=f1)
show_pil_image(pred_image)
